In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math as m

from ipywidgets import interact, FloatSlider, IntSlider, Play, jslink, VBox, fixed

In [2]:
#Cambell
period = 100*60*60*24*365.25
periastron_passage_time_quotient = 0.4
eccentricity = 0.5
semi_major_axis = 149597870700 
inclination = 60 * np.pi/180
longitude_periastron = 250 * np.pi/180
angle_of_ascending_node = 120 * np.pi/180

num_periods = 1
mass_quotient = 1 

$x_{n+1} = x_{n} \frac{f(xn)}{f'(xn)}$

In [3]:
def trobarE(t,periastron_passage_time,eccentricity,mu):
    #Newton-Raphson
    #x_{n+1} = x_{n} \frac{f(xn)}{f'(xn)} 
    
    Eo = 0
    E = 1  
    ε = 1e-5

    M = mu*(t-periastron_passage_time)

    def Ef (M,Ei,eccentricity):
        return (Ei - eccentricity*m.sin(Ei) - M)/(1 - eccentricity*m.cos(Ei))
    
    while abs(E-Eo) > ε:
        Eo = E
        E = Eo - Ef(M,Eo,eccentricity)
        


    return E
        

# Representació de l'òrbita en 2D.

In [4]:
def orbites (num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    t = 0
    dt = period*1e-3
    periastron_passage_time = period*periastron_passage_time_quotient
    n = int(period * num_periods / dt)
    mu = 2*np.pi/period
    E = 0
    ε = 1e-10
    G = 6.67 * 1e-11
    cnst = 4*np.pi**2*semi_major_axis**3/(G*period**2)
    m1 = cnst/(1+mass_quotient)
    m2 = mass_quotient*m1
    x1_list = np.zeros(n)
    y1_list = np.zeros(n)
    x2_list = np.zeros(n)
    y2_list = np.zeros(n)
    xcm_list = np.zeros(n)
    ycm_list = np.zeros(n)
    t_list= np.zeros(n)
    i=0
    t=0


    argument_of_periapsis = longitude_periastron - angle_of_ascending_node

    
    c1 = m2/(m1 + m2)
    c2 = - m1/(m1 + m2)

    if inclination == 0:
        angle_of_ascending_node = 0

    if eccentricity == 0:
         longitude_periastron = 0

    A = semi_major_axis*(np.cos(angle_of_ascending_node)*np.cos(argument_of_periapsis) - np.sin(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    B = semi_major_axis*(np.sin(angle_of_ascending_node)*np.cos(argument_of_periapsis) + np.cos(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    F = semi_major_axis*(-np.cos(angle_of_ascending_node)*np.sin(argument_of_periapsis) - np.sin(angle_of_ascending_node)*np.cos(argument_of_periapsis)*np.cos(inclination))
    G = semi_major_axis*(-np.sin(angle_of_ascending_node)*np.sin(argument_of_periapsis) + np.cos(angle_of_ascending_node)*np.cos(argument_of_periapsis)*np.cos(inclination))


 
    xf = A*(eccentricity)
    yf = B*(eccentricity)

    

                
    for i in range(n):
        E = trobarE(t,periastron_passage_time,eccentricity,mu)

        X = np.cos(E) - eccentricity
        Y = np.sqrt(1-eccentricity**2) * np.sin(E)


        x = A*X + F*Y
        y = B*X + G*Y

        x1 = c1*x
        x2 = c2*x
        y1 = c1*y
        y2 = c2*y        

        xcm = x2*c1 - x1*c2
        ycm = y2*c1 - y1*c2


        xcm_list[i]=xcm
        ycm_list[i]=ycm
        x1_list[i]=x1
        y1_list[i]=y1
        x2_list[i]=x2
        y2_list[i]=y2
        t_list[i]=t

        t = t+dt



    return xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf

In [5]:
def plot2D(frame,num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):

    xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf = orbites(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)
    
    
    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.grid(True)
    plt.show()


num_periods = float(input('Nombre de periodes:'))
period = float(input("Determina el periòde de l'òrbita en anys:"))*60*60*24*365.25

xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf = orbites(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)


# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)




jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot2D, frame=slider,
               num_periods=fixed(1),
    period = fixed(period),
    periastron_passage_time_quotient = FloatSlider(min=0, max=1, value=periastron_passage_time_quotient),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.01, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient))

display(ui, out)

Nombre de periodes: 1
Determina el periòde de l'òrbita en anys: 100


interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.4, description='pe…

<function __main__.plot2D(frame, num_periods, period, periastron_passage_time_quotient, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>

# Representació de l'òrbita en 3D.

In [6]:
def orbita3D(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    t = 0
    dt = period*1e-3
    periastron_passage_time = periastron_passage_time_quotient*period
    n = int(period * num_periods / dt)
    mu = 2*np.pi/period
    E = 0
    ε = 1e-10
    G = 6.67 * 1e-11
    cnst = 4*np.pi**2*semi_major_axis**3/(G*period**2)
    m1 = cnst/(1+mass_quotient)
    m2 = m1*mass_quotient
    zcm_list = np.zeros(n)
    z1_list = np.zeros(n)
    z2_list = np.zeros(n)
    i=0

    argument_of_periapsis = longitude_periastron - angle_of_ascending_node
    
    
    c1 = m2/(m1 + m2)
    c2 = - m1/(m1 + m2)
    
    
    
    xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf = orbites(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    if inclination == 0:
        angle_of_ascending_node = 0

    if eccentricity == 0:
         longitude_periastron = 0

    H = semi_major_axis*np.sin(argument_of_periapsis)*np.sin(inclination)
    K = semi_major_axis*np.cos(argument_of_periapsis)*np.sin(inclination)
    
    zf = H*(eccentricity)
    
            
    while t < period*num_periods:
        E = trobarE(t,periastron_passage_time,eccentricity,mu)

        X = np.cos(E) - eccentricity
        Y = np.sqrt(1-eccentricity**2) * np.sin(E)
        
        z = H*X + K*Y

        z1 = c1*z
        z2 = c2*z 
        zcm = z2*c1 - z1*c2

        zcm_list[i]=zcm
        z1_list[i]=z1
        z2_list[i]=z2

        i += 1
        t = t+dt

    
    return xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf

In [7]:
def plot_orbites3D(frame,num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)
    

    ax.set_box_aspect([1,1,1])
    plt.show()

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.grid(True)
    plt.show()


num_periods = float(input('Nombre de periodes:'))
period = float(input("Determina el periòde de l'òrbita en anys:"))*60*60*24*365.25

xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)


# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_orbites3D, frame=slider,
               num_periods=fixed(num_periods),
    period = fixed(period),
    periastron_passage_time_quotient = FloatSlider(min=0, max=1, value=periastron_passage_time_quotient),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient))

display(ui, out)

Nombre de periodes: 1
Determina el periòde de l'òrbita en anys: 100


interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.4, description='pe…

<function __main__.plot_orbites3D(frame, num_periods, period, periastron_passage_time_quotient, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>

# Representació de l'orbita 3D en fucnió de les coordenades esfèriques.

In [8]:
def plot3Dpolars(frame,phi,theta,num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,mass_quotient):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,0,0,0,mass_quotient)
    fig = plt.figure(figsize=(6,6))
    
    ax = fig.add_subplot(111, projection='3d')
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    
    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)
    
    ax.view_init(elev=np.degrees(theta), azim=np.degrees(phi))
    ax.set_box_aspect([1,1,1])
    plt.grid(True)
    
    plt.show()
    
    # widgets de reproducció
num_periods = float(input('Nombre de periodes:'))
period = float(input("Determina el periòde de l'òrbita en anys:"))*60*60*24*365.25
    
play = Play( value=0, min=0, max=999, step=1, interval=30 )
phi_slider = FloatSlider(min=0, max=2*np.pi, step=0.01, value=0, description='phi')
theta_slider = FloatSlider(min=0, max=np.pi, step=0.01, value=np.pi/4, description='theta')
jslink((play, 'value'), (slider, 'value'))
ui = VBox([play, slider])

out = interact(plot3Dpolars, frame=slider, phi=phi_slider, theta=theta_slider,
               num_periods=fixed(num_periods),
               period = fixed(period),
               periastron_passage_time_quotient = FloatSlider(min=0, max=1, value=periastron_passage_time_quotient),
               eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
               semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
               inclination = fixed(inclination),
               longitude_periastron = fixed(longitude_periastron),
               angle_of_ascending_node = fixed(angle_of_ascending_node),
               mass_quotient = fixed(mass_quotient))

display(ui, out)

Nombre de periodes: 1
Determina el periòde de l'òrbita en anys: 100


interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.0, description='ph…

<function __main__.plot3Dpolars(frame, phi, theta, num_periods, period, periastron_passage_time_quotient, eccentricity, semi_major_axis, mass_quotient)>

# Càlcul dels vectors del moment angular i eccentricitat.

In [9]:
def eccentricity_vector(eccentricity,inclination,longitude_periastron,angle_of_ascending_node):
    argument_of_periapsis = longitude_periastron - angle_of_ascending_node
    ex = eccentricity*(np.cos(angle_of_ascending_node)*np.cos(argument_of_periapsis) - np.sin(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    ey = eccentricity*(np.sin(angle_of_ascending_node)*np.cos(argument_of_periapsis) + np.cos(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    ez = eccentricity*np.sin(inclination)*np.sin(argument_of_periapsis)  

    return ex, ey, ez

In [10]:
def moment_angular(period,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    G = 6.67 * 1e-11
    cnst = 4*np.pi**2*semi_major_axis**3/(G*period**2)
    m1 = cnst/(1+mass_quotient)
    m2 = mass_quotient*m1
    h = m1*m2*((G**2 * period)/((m1+m2)*2*np.pi))**(1/3)/1e26


    hx = h*np.sin(angle_of_ascending_node)*np.sin(inclination)
    hy = - h*np.cos(angle_of_ascending_node)*np.sin(inclination)
    hz = h*np.cos(inclination)

    return hx, hy, hz

In [11]:
def plot3Dvectors(frame,num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)

    ex, ey, ez = eccentricity_vector(eccentricity, inclination, longitude_periastron, angle_of_ascending_node)
    hx, hy, hz = moment_angular(period, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)
    x0, y0, z0 = xcm_list[-1], ycm_list[-1], zcm_list[-1]
    
    x0, y0, z0 = xcm_list[-1], ycm_list[-1], zcm_list[-1]
    ax.plot([x0, x0 + ex*semi_major_axis], [y0, y0 + ey*semi_major_axis], [z0, z0 + ez*semi_major_axis], color='red', linewidth=2, label='Excentricity vector')
    ax.plot([x0, x0 + hx/1000], [y0, y0 + hy/1000], [z0, z0 + hz/1000], color='green', linewidth=2, label='Angular momentum')
    

    ax.set_box_aspect([1,1,1])
    ax.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)
    plt.plot([x0, x0 + ex*semi_major_axis], [y0, y0 + ey*semi_major_axis], color='red', linewidth=2, label='Excentricity vector')
    plt.plot([x0, x0 + hx/1000], [y0, y0 + hy/1000], color='green', linewidth=2, label='Angular momentum')

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.legend()
    plt.grid(True)
    plt.show()

num_periods = float(input('Nombre de periodes:'))
period = float(input("Determina el periòde de l'òrbita en anys:"))*60*60*24*365.25


# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=999,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=999,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot3Dvectors, frame=slider,
               num_periods=fixed(num_periods),
    period = fixed(period),
    periastron_passage_time_quotient = FloatSlider(min=0, max=1, value=periastron_passage_time_quotient),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient))

display(ui, out)

Nombre de periodes: 1
Determina el periòde de l'òrbita en anys: 100


interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.4, description='pe…

<function __main__.plot3Dvectors(frame, num_periods, period, periastron_passage_time_quotient, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>

# Visió del sistema des de la Terra.

In [12]:
def plot_Terra(frame,num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient, zoom):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)

    ex, ey, ez = eccentricity_vector(eccentricity, inclination, longitude_periastron, angle_of_ascending_node)
    hx, hy, hz = moment_angular(period, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)
    x0, y0, z0 = xcm_list[-1], ycm_list[-1], zcm_list[-1]
    
    x0, y0, z0 = xcm_list[-1], ycm_list[-1], zcm_list[-1]
    ax.plot([x0, x0 + ex*semi_major_axis], [y0, y0 + ey*semi_major_axis], [z0, z0 + ez*semi_major_axis], color='red', linewidth=2, label='Excentricity vector')
    ax.plot([x0, x0 + hx/1000], [y0, y0 + hy/1000], [z0, z0 + hz/1000], color='green', linewidth=2, label='Angular momentum')
    

    ax.set_box_aspect([1,1,1])
    lim = semi_major_axis * 2 / zoom
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.view_init(elev=90, azim=-90)
    ax.legend()
    plt.show()

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)
    plt.plot([x0, x0 + ex*semi_major_axis], [y0, y0 + ey*semi_major_axis], color='red', linewidth=2, label='Excentricity vector')
    plt.plot([x0, x0 + hx/1000], [y0, y0 + hy/1000], color='green', linewidth=2, label='Angular momentum')

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.xlim(-lim, lim)
    plt.ylim(-lim, lim)
    plt.legend()
    plt.grid(True)
    plt.show()

num_periods = float(input('Nombre de periodes:'))
period = float(input("Determina el periòde de l'òrbita en anys:"))*60*60*24*365.25


# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=999,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=999,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_Terra, frame=slider,
               num_periods=fixed(num_periods),
    period = fixed(period),
    periastron_passage_time_quotient = FloatSlider(min=0, max=1, value=periastron_passage_time_quotient),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient),
    zoom = FloatSlider(min=0.1, max=5, step=0.1, value=1))

display(ui, out)



Nombre de periodes: 1
Determina el periòde de l'òrbita en anys: 100


interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.4, description='pe…

<function __main__.plot_Terra(frame, num_periods, period, periastron_passage_time_quotient, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient, zoom)>

# Stellar winds

In [13]:
mass_loss = 1e23/(365.25*60**2*24)

In [16]:
def setllar_winds (num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient, mass_loss):
    t = 0
    dt = period*1e-3
    periastron_passage_time=periastron_passage_time_quotient*period
    N = int(num_periods*period/dt)
    E = 0
    ε = 1e-10
    G = 6.67 * 1e-11
    cnst = 4*np.pi**2*semi_major_axis**3/(G*period**2)
    m1 = cnst/(1+mass_quotient)
    m2 = mass_quotient*m1
    mass_T = m1 + m2 # massa inicial
    x1_list = np.zeros(N)
    y1_list = np.zeros(N)
    z1_list = np.zeros(N)
    x2_list = np.zeros(N)
    y2_list = np.zeros(N)
    z2_list = np.zeros(N)
    xcm_list = np.zeros(N)
    ycm_list = np.zeros(N)
    zcm_list = np.zeros(N)
    xf_list = np.zeros(N)
    yf_list = np.zeros(N)
    zf_list = np.zeros(N)
    mq_list= np.zeros(N)
    t_list= np.zeros(N)
    i=0



    p0 = period #guardar el valor inicial del periode
    a0 = semi_major_axis #guardar el valor inicial de semi eix major



    argument_of_periapsis = longitude_periastron - angle_of_ascending_node



    if inclination == 0:
        angle_of_ascending_node = 0

    if eccentricity == 0:
         longitude_periastron = 0


    m1t_list = np.zeros(N)
    MT_list = np.zeros(N)
    semi_major_axis_list = np.zeros(N)
    period_list = np.zeros(N)
    


    for j in range(N):
        t = j*dt
        m1t_list[j]=m1 - mass_loss*t
        MT_list[j] = m1t_list[j] + m2
        semi_major_axis_list[j] = a0*(MT_list[j]/mass_T)**(-mass_loss/mass_T)
        period_list[j] = p0*(MT_list[j]/mass_T)**(-2*mass_loss/mass_T)        
    

    t=0

    A = semi_major_axis*(np.cos(angle_of_ascending_node)*np.cos(argument_of_periapsis) - np.sin(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    B = semi_major_axis*(np.sin(angle_of_ascending_node)*np.cos(argument_of_periapsis) + np.cos(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    F = semi_major_axis*(-np.cos(angle_of_ascending_node)*np.sin(argument_of_periapsis) - np.sin(angle_of_ascending_node)*np.cos(argument_of_periapsis)*np.cos(inclination))
    G = semi_major_axis*(-np.sin(angle_of_ascending_node)*np.sin(argument_of_periapsis) + np.cos(angle_of_ascending_node)*np.cos(argument_of_periapsis)*np.cos(inclination))
    H = semi_major_axis*np.sin(argument_of_periapsis)*np.sin(inclination)
    K = semi_major_axis*np.cos(argument_of_periapsis)*np.sin(inclination)
 
    xf = A*(eccentricity)
    yf = B*(eccentricity)
    zf = H*(eccentricity)

    
    while t < p0*num_periods:

        m1t = m1t_list[i]
        semi_major_axis = semi_major_axis_list[i]
        period = period_list[i]
        
            
        mu = 2*np.pi/period
        periastron_passage_time = period*periastron_passage_time_quotient

                
        c1 = m2/(m1t + m2)
        c2 = - m1t/(m1t + m2)
        
        E = trobarE(t,periastron_passage_time,eccentricity,mu)

        X = np.cos(E) - eccentricity
        Y = np.sqrt(1-eccentricity**2) * np.sin(E)


        x = A*X + F*Y
        y = B*X + G*Y
        z = H*X + K*Y
        

        x1 = c1*x
        x2 = c2*x
        y1 = c1*y
        y2 = c2*y    
        z1 = c1*z
        z2 = c2*z 

        xcm = x2*c1 - x1*c2
        ycm = y2*c1 - y1*c2
        zcm = z2*c1 - z1*c2
        


        xcm_list[i]=xcm
        ycm_list[i]=ycm
        zcm_list[i]=zcm
        x1_list[i]=x1
        y1_list[i]=y1
        z1_list[i]=z1
        x2_list[i]=x2
        y2_list[i]=y2
        z2_list[i]=z2
        xf_list[i]=xf
        yf_list[i]=yf
        zf_list[i]=zf
        mq_list[i]=m1t/m1
        t_list[i]=t

        t = t+dt
        i=i+1


        if t >= m1/mass_loss:
            break





    return xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf_list, yf_list, zf_list, mq_list

In [17]:
def plot_stellar_wind(frame,num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient,mass_loss):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf_list, yf_list, zf_list, mq_list = setllar_winds (num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient, mass_loss)

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf_list[frame], yf_list[frame], zf_list[frame], color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100*mq_list[frame])
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)
    

    ax.set_box_aspect([1,1,1])
    plt.show()

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    
    # punts actuals
    plt.scatter (xf_list[frame], yf_list[frame], color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15*mq_list[frame])
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.grid(True)
    plt.show()

num_periods = float(input('Nombre de periodes:'))
period = float(input("Determina el periòde de l'òrbita en anys:"))*60*60*24*365.25

    

xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf_list, yf_list, zf_list, mq_list = setllar_winds (num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient, mass_loss)

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_stellar_wind, frame=slider,
               num_periods=fixed(num_periods),
    period = fixed(period),
    periastron_passage_time_quotient = FloatSlider(min=0, max=1, value=periastron_passage_time_quotient),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient),
    mass_loss = FloatSlider(min=0, max=1e24/(365.25*60**2*24), value=mass_loss))

display(ui, out)



Nombre de periodes: 1.5
Determina el periòde de l'òrbita en anys: 100


interactive(children=(IntSlider(value=0, description='frame', max=1499), FloatSlider(value=0.4, description='p…

<function __main__.plot_stellar_wind(frame, num_periods, period, periastron_passage_time_quotient, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient, mass_loss)>

# Gravitational waves

In [38]:
def gravitational_waves (num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    t = 0
    dt = period*1e-3
    periastron_passage_time=periastron_passage_time_quotient*period
    N = int(num_periods*period/dt)
    E = 0
    ε = 1e-10
    G = 6.67 * 1e-11
    cnst = 4*np.pi**2*semi_major_axis**3/(G*period**2)
    m1 = cnst/(1+mass_quotient)
    m2 = mass_quotient*m1
    mT = m1 + m2 
    x1_list = np.zeros(N)
    y1_list = np.zeros(N)
    z1_list = np.zeros(N)
    x2_list = np.zeros(N)
    y2_list = np.zeros(N)
    z2_list = np.zeros(N)
    xcm_list = np.zeros(N)
    ycm_list = np.zeros(N)
    zcm_list = np.zeros(N)
    xf_list = np.zeros(N)
    yf_list = np.zeros(N)
    zf_list = np.zeros(N)
    t_list= np.zeros(N)
    i=0



    p0 = period #guardar el valor inicial del periode
    a0 = semi_major_axis #guardar el valor inicial de semi eix major

    c = 3 * 1e8

    cnst_a = 32/5 * G**3/c**5 * m1*m2*mT #constnat emprada per a resoldre la edo del semieix major
    cnst_p = 32/5 * G**(5/3)/c**5 * m1*m2/mT**(1/3) * (2*np.pi)**8/3 #constant emprada per a resoldre la edo del periode



    argument_of_periapsis = longitude_periastron - angle_of_ascending_node



    if inclination == 0:
        angle_of_ascending_node = 0

    if eccentricity == 0:
         longitude_periastron = 0


    semi_major_axis_list = np.zeros(N)
    period_list = np.zeros(N)
    


    for j in range(N):
        t = j*dt
        semi_major_axis_list[j] = (a0**4 - 3/2 * t/cnst_a)**(1/4)
        period_list[j] = (p0**(8/3) - t/(3*cnst_p))**(3/8)
    

    t=0

    A = semi_major_axis*(np.cos(angle_of_ascending_node)*np.cos(argument_of_periapsis) - np.sin(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    B = semi_major_axis*(np.sin(angle_of_ascending_node)*np.cos(argument_of_periapsis) + np.cos(angle_of_ascending_node)*np.sin(argument_of_periapsis)*np.cos(inclination))
    F = semi_major_axis*(-np.cos(angle_of_ascending_node)*np.sin(argument_of_periapsis) - np.sin(angle_of_ascending_node)*np.cos(argument_of_periapsis)*np.cos(inclination))
    G = semi_major_axis*(-np.sin(angle_of_ascending_node)*np.sin(argument_of_periapsis) + np.cos(angle_of_ascending_node)*np.cos(argument_of_periapsis)*np.cos(inclination))
    H = semi_major_axis*np.sin(argument_of_periapsis)*np.sin(inclination)
    K = semi_major_axis*np.cos(argument_of_periapsis)*np.sin(inclination)
 
    xf = A*(eccentricity)
    yf = B*(eccentricity)
    zf = H*(eccentricity)

            
    c1 = m2/(m1 + m2)
    c2 = - m1/(m1 + m2)
                
    while t < p0*num_periods:

        semi_major_axis = semi_major_axis_list[i]
        period = period_list[i]
        
            
        mu = 2*np.pi/period
        periastron_passage_time = period*periastron_passage_time_quotient

        
        E = trobarE(t,periastron_passage_time,eccentricity,mu)

        X = np.cos(E) - eccentricity
        Y = np.sqrt(1-eccentricity**2) * np.sin(E)


        x = A*X + F*Y
        y = B*X + G*Y
        z = H*X + K*Y
        

        x1 = c1*x
        x2 = c2*x
        y1 = c1*y
        y2 = c2*y    
        z1 = c1*z
        z2 = c2*z 

        xcm = x2*c1 - x1*c2
        ycm = y2*c1 - y1*c2
        zcm = z2*c1 - z1*c2
        


        xcm_list[i]=xcm
        ycm_list[i]=ycm
        zcm_list[i]=zcm
        x1_list[i]=x1
        y1_list[i]=y1
        z1_list[i]=z1
        x2_list[i]=x2
        y2_list[i]=y2
        z2_list[i]=z2
        xf_list[i]=xf
        yf_list[i]=yf
        zf_list[i]=zf
        t_list[i]=t

        t = t+dt
        i=i+1


        if t >= p0**(8/3)*cnst_p*3:
            break



    return xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf_list, yf_list, zf_list

In [39]:
def plot_gravitational_waves(frame,num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf_list, yf_list, zf_list = gravitational_waves (num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf_list[frame], yf_list[frame], zf_list[frame], color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)
    

    ax.set_box_aspect([1,1,1])
    plt.show()

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    
    # punts actuals
    plt.scatter (xf_list[frame], yf_list[frame], color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.grid(True)
    plt.show()

num_periods = float(input('Nombre de periodes:'))
period = float(input("Determina el periòde de l'òrbita en anys:"))*60*60*24*365.25

    

xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf_list, yf_list, zf_list = gravitational_waves (num_periods,period,periastron_passage_time_quotient,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_gravitational_waves, frame=slider,
               num_periods=fixed(num_periods),
    period = fixed(period),
    periastron_passage_time_quotient = FloatSlider(min=0, max=1, value=periastron_passage_time_quotient),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient))

display(ui, out)



Nombre de periodes: 1
Determina el periòde de l'òrbita en anys: 100


interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.4, description='pe…

<function __main__.plot_gravitational_waves(frame, num_periods, period, periastron_passage_time_quotient, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>